In [1]:
import os
import sys
sys.path.append('../..')
import utils

import panel as pn #GUI
import ollama

pn.extension()

In [5]:

def get_completion_from_messages(
    messages,
    model="llama3.2:1b",
    temperature=0,
    max_tokens=500
):
    formatted_messages = [
        {"role": msg["role"], "content": msg["content"]}
        for msg in messages
    ]

    try:
        response = ollama.chat(
            model=model,
            messages=formatted_messages,
            format="json",
            options={
                "temperature": temperature,
                "num_predict": max_tokens
            }
        )

        return response["message"]["content"].strip()

    except Exception as e:
        print("ERROR: Ollama API call failed:", str(e))
        return ""


In [18]:

def evaluate_response(user_input, final_response, product_information):

    evaluation_messages = [
        {
            "role": "system",
            "content": """
You are a response evaluator.

Your only job is to check whether an assistant response
correctly answers a customer's question using the
provided product information.

You must return exactly one character:

Y = The answer is correct.
N = The answer is incorrect or contains invented information.

Do not generate product information.
Do not generate JSON.
Do not answer the customer.
Do not provide explanations.

Your entire response must be either Y or N.
"""
        },
        {
            "role": "user",
            "content": f"""
Customer question:
{user_input}

Available product information:
{product_information}

Assistant response:
{final_response}

Is the assistant response correct?

Answer:
"""
        }
    ]

    try:
        response = ollama.chat(
            model="llama3.2:1b",
            messages=evaluation_messages,
            options={
                "temperature": 0,
                "num_predict": 5,
                "stop": ["\n"]
            }
        )

        result = response["message"]["content"].strip().upper()

        print("Raw evaluator response:", repr(result))

        if result.startswith("Y"):
            return "Y"

        if result.startswith("N"):
            return "N"

        return None

    except Exception as e:
        print("Evaluation error:", str(e))
        return None


In [27]:
test_result = evaluate_response(
    "Tell me about SmartX ProPhone.",
    "SmartX ProPhone has 128GB storage and a 12MP dual camera.",
    """
    Product: SmartX ProPhone
    Storage: 128GB
    Camera: 12MP dual camera
    """
)

print("Evaluation:", test_result)

Raw evaluator response: 'N'
Evaluation: N


In [34]:

def generate_customer_response(user_input, product_information):

    messages = [
        {
            "role": "system",
            "content": """
You are a professional electronics store customer service assistant.

Your task is to answer the customer's entire question.

STRICT INSTRUCTIONS:

1. Read the complete customer question.
2. Read ALL the supplied product information.
3. Identify every product the customer asked about.
4. Provide information about EVERY requested product.
5. Never ignore a requested product.
6. Never invent specifications or prices.
7. Use only the supplied product information.
8. Do not return JSON.
9. Use friendly conversational language.
10. If information is unavailable, say so clearly.

Structure your answer with separate headings for each product.
"""
        },
        {
            "role": "user",
            "content": f"""
CUSTOMER QUESTION:
{user_input}

PRODUCT DATABASE:
{product_information}

Answer ALL parts of the customer's question.
"""
        }
    ]

    response = get_completion_from_messages(messages)

    return response


In [37]:
def process_user_message(user_input,all_messages,debug=True):
    delimiter="```"

    #Step 1 : Input vaildation
    if debug:
        print("Step 1 : Input passed basic check.")

    #Step 2: Extract products and categories 
    try:
        category_and_product_response=utils.find_category_and_product_only(
            user_input,utils.get_products_and_category()
        )
        category_and_product_list=utils.read_string_to_list(category_and_product_response)
    except Exception as e:
        category_and_product_list=[]
        if debug:
            print(f"Step 2 Error: Failed to extract categories /products -{e}")

        if debug:
            print(f"Step 2 Error : Failed to extract categories/products - {e}")

    # Step 3 : Fetch product information
    product_information = utils.generate_output_string(category_and_product_list) if category_and_product_list else "No Matching products found."

    if debug:
        print("Step 3 : Retrived product information.")
        
    print("\n========== PRODUCT INFORMATION ==========")
    print(product_information)
    print("=========================================")
    
    # Step 4 : Generate customer service response
    # Step 4: Generate customer response

    final_response = generate_customer_response(
        user_input,
        product_information
    )

    print("\n========== CUSTOMER RESPONSE ==========")
    print(final_response)
    print("=======================================\n")

    all_messages.append({
    'role': 'assistant',
    'content': final_response
    })
    final_response = get_completion_from_messages(all_messages)
    all_messages.append({'role':'assistant','content':final_response})

    if debug:
        print('Step 4 : Generated response')

    final_response = get_completion_from_messages(all_messages)

    print("\n========== CUSTOMER RESPONSE ==========")
    print(final_response)
    print("=======================================\n")

    #Step 5 : Evaluate response quality

    # Step 6: Evaluate response quality

    cleaned_evaluation_response = evaluate_response(
        user_input,
        final_response,
        product_information
    )



    print("Step 6: Evaluation result:",
          cleaned_evaluation_response)


    # Step 7: Decision based on evaluation

    if cleaned_evaluation_response == "Y":

        print("Step 7: Response is approved")
        return final_response, all_messages

    elif cleaned_evaluation_response == "N":

        print("Step 7: Response is not sufficient")

        return (
            "I'm unable to provide the information you're looking for. "
            "Let me connect you with a representative.",
            all_messages
        )

    else:

        print("Step 7: Unexpected evaluation response")

        return (
            "There was an issue processing your request. Please try again.",
            all_messages
        )

#Test the Function
user_input="Tell me about the SmartX Pro Phone and the FotoSnap Camera , the DSLR one. Also , what tell me about your TVs?"
response,_=process_user_message(user_input,[])
    
print(response)

Step 1 : Input passed basic check.
Raw Ollama response: '[\n    {\n        "category": "Smartphones and Accessories",\n        "products": ["SmartX ProPhone"]\n    },\n    {\n        "category": "Cameras and Camcorders",\n        "products": ["FotoSnap DSLR Camera"]\n    },\n    {\n        "category": "Cameras and Camcorders",\n        "products": ["ActionCam 4K"]\n    }\n]'
Step 3 : Retrived product information.

========== PRODUCT INFORMATION ==========
{
    "name": "SmartX ProPhone",
    "category": "Smartphones and Accessories",
    "brand": "SmartX",
    "model_number": "SX-PP10",
    "warranty": "1 year",
    "rating": 4.6,
    "features": [
        "6.1-inch display",
        "128GB storage",
        "12MP dual camera",
        "5G"
    ],
    "description": "A powerful smartphone with advanced camera features.",
    "price": 899.99
}
{
    "name": "FotoSnap DSLR Camera",
    "category": "Cameras and Camcorders",
    "brand": "FotoSnap",
    "model_number": "FS-DSLR200",
    "w

In [36]:
import importlib
import utils

importlib.reload(utils)

products = utils.get_products_and_category()

response = utils.find_category_and_product_only(
    "Tell me about SmartX ProPhone and FotoSnap DSLR Camera. Also tell me about your TVs.",
    products
)

print("RAW RESPONSE:", repr(response))

result = utils.read_string_to_list(response)

print("EXTRACTED PRODUCTS:", result)

Raw Ollama response: '[ {\n  "category": "Smartphones and Accessories",\n  "products": ["SmartX ProPhone"]\n} ]\n[ {\n  "category": "Cameras and Camcorders",\n  "products": ["FotoSnap DSLR Camera"]\n} ]\n[ {\n  "category": "Televisions and Home Theater Systems",\n  "products": ["CineView 4K TV"]\n} ]'
RAW RESPONSE: '[ {\n  "category": "Smartphones and Accessories",\n  "products": ["SmartX ProPhone"]\n} ]\n[ {\n  "category": "Cameras and Camcorders",\n  "products": ["FotoSnap DSLR Camera"]\n} ]\n[ {\n  "category": "Televisions and Home Theater Systems",\n  "products": ["CineView 4K TV"]\n} ]'
JSON parsing error: Extra data: line 5 column 1 (char 87)
Actual response: '[ {\n  "category": "Smartphones and Accessories",\n  "products": ["SmartX ProPhone"]\n} ]\n[ {\n  "category": "Cameras and Camcorders",\n  "products": ["FotoSnap DSLR Camera"]\n} ]\n[ {\n  "category": "Televisions and Home Theater Systems",\n  "products": ["CineView 4K TV"]\n} ]'
EXTRACTED PRODUCTS: []


In [38]:
pn.extension(raw_css=['''
.assistant-response {
    background-color: #F6F6F6;
    padding: 10px;
    border-radius: 5px;
}
'''])

In [40]:
def collect_messages(debug=False):
    user_input = inp.value
    if debug: print(f"User Input = {user_input}")
    if user_input == "":
        return
    inp.value = ''
    global context

    response, context = process_user_message(user_input, context, debug=False)
    context.append({'role': 'assistant', 'content': f"{response}"})

    panels.append(pn.Row('User:', pn.pane.Markdown(user_input, width=600)))
    panels.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600, css_classes=['assistant-response'])))

    return pn.Column(*panels)

panels = []  # collect display
context = [{'role': 'system', 'content': "You are a Service Assistant"}]

inp = pn.widgets.TextInput(placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Service Assistant")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard


Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(label='Service Assistant', name='Service Assistant')
    [2] ParamFunction(function, _pane=Str, defer_load=False, height=300, loading_indicator=True)